In [3]:
import sys
sys.path.insert(0, '..')
from dependencies import *

envelopes_log = eelbrain.load.unpickle(PROCESSED_PREDICTOR_DIR / f'~processed_envelopes-log.pickle')
subject_model_predictors = eelbrain.load.unpickle(PREDICTOR_DIR / f'~concatenated_predictors.pickle')
durations = get_durations(envelopes_log)
models = get_models(exclude=['envelope_log_8band'])

In [16]:
# ------------------------------------------------
# Directories
# ------------------------------------------------
# Decoder evaluation (statistics only)
# ------------------------------------------------
r_values = {}
r2_values = {}
rmse_values = {model: [] for model in models}

for model in models:
    r_values[model] = []
    r2_values[model] = []

for subject in SUBJECTS:

    print(f'\n===== Subject {subject} =====')

    eeg = eelbrain.load.unpickle(STIMULUS_DIR / f'{subject}concatenated_eeg.pickle')

    for model in models:

        print(f'\nModel: {model}')

        # Load decoder TRF
        trf_decoder = eelbrain.load.unpickle(
            TRF_DIR / subject / f'{subject} decoder-{model}.pickle'
        )

        predictors = subject_model_predictors[subject][model]

        # Predict stimulus from EEG
        predicted = eelbrain.convolve(trf_decoder.h_scaled, eeg)

        # Build actual stimulus (sum if multiple)
        stimulus = sum(predictors) if len(predictors) > 1 else predictors[0]

        # Convert to numpy arrays
        env = stimulus.x
        pred = predicted.x

        # Correlation and r²
        r = np.corrcoef(env, pred)[0,1]
        r2 = r**2

        r_values[model].append(r)
        r2_values[model].append(r2)

        # ----------------------------------------
        # RMSE (normalized ONLY here)
        # ----------------------------------------
        env_norm = (env - np.mean(env)) / np.std(env)
        pred_norm = (pred - np.mean(pred)) / np.std(pred)

        rmse = np.sqrt(np.mean((env_norm - pred_norm)**2))

        rmse_values[model].append(rmse)

        print(f'Normalized RMSE = {rmse:.4f}')

        print(f'r = {r:.4f}, r² = {r2:.4f}, RMSE = {rmse:.4f}')



===== Subject S05 =====

Model: envelope_log
Normalized RMSE = 1.2933
r = 0.1637, r² = 0.0268, RMSE = 1.2933

Model: envelope_onset
Normalized RMSE = 1.3368
r = 0.1064, r² = 0.0113, RMSE = 1.3368

Model: envelope_log_onset
Normalized RMSE = 1.2933
r = 0.1637, r² = 0.0268, RMSE = 1.2933

===== Subject S34 =====

Model: envelope_log
Normalized RMSE = 1.2437
r = 0.2266, r² = 0.0514, RMSE = 1.2437

Model: envelope_onset
Normalized RMSE = 1.3210
r = 0.1275, r² = 0.0163, RMSE = 1.3210

Model: envelope_log_onset
Normalized RMSE = 1.2437
r = 0.2266, r² = 0.0514, RMSE = 1.2437

===== Subject S35 =====

Model: envelope_log
Normalized RMSE = 1.2451
r = 0.2249, r² = 0.0506, RMSE = 1.2451

Model: envelope_onset
Normalized RMSE = 1.3564
r = 0.0801, r² = 0.0064, RMSE = 1.3564

Model: envelope_log_onset
Normalized RMSE = 1.2451
r = 0.2249, r² = 0.0506, RMSE = 1.2451

===== Subject S03 =====

Model: envelope_log
Normalized RMSE = 1.2408
r = 0.2302, r² = 0.0530, RMSE = 1.2408

Model: envelope_onset
Nor

In [5]:
for model in r_values:
    mean_r = np.mean(r_values[model])
    std_r = np.std(r_values[model])
    mean_r2 = np.mean(r2_values[model])
    std_r2 = np.std(r2_values[model])
    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f}, r² = {mean_r2:.4f} ± {std_r2:.4f}')

envelope_log: r = 0.2094 ± 0.0830, r² = 0.0507 ± 0.0302
envelope_onset: r = 0.0850 ± 0.0361, r² = 0.0085 ± 0.0051
envelope_log_onset: r = 0.2094 ± 0.0830, r² = 0.0507 ± 0.0302


In [6]:
t_stat, p_val = ttest_rel(r_values['envelope_log'], r_values['envelope_onset'])
print(f'Envelope log vs. onset: t={t_stat:.2f}, p={p_val:.4f}')

Envelope log vs. onset: t=13.08, p=0.0000


In [ ]:
# ------------------------------------------------
# Storage
# ------------------------------------------------
r_values_universal = {model: [] for model in models}
r2_values_universal = {model: [] for model in models}
rmse_values = {model: [] for model in models}

# ------------------------------------------------
# Loop over models
# ------------------------------------------------
for model in models:

    print(f'\n===== Universal decoder: {model} =====')

    trf_universal = eelbrain.load.unpickle(
        TRF_DIR / f'decoder-universal-trf-{model}.pickle'
    )

    for subject in SUBJECTS:

        print(f'\nSubject: {subject}')

        eeg = eelbrain.load.unpickle(
            STIMULUS_DIR / f'{subject}concatenated_eeg.pickle'
        )

        predictors = subject_model_predictors[subject][model]

        predicted = eelbrain.convolve(trf_universal, eeg)

        stimulus = predictors[0]

        env = stimulus.x
        pred = predicted.x

        # ----------------------------------------
        # Correlation
        # ----------------------------------------
        r = np.corrcoef(env, pred)[0,1]
        r2 = r**2

        r_values_universal[model].append(r)
        r2_values_universal[model].append(r2)

        print(f'r = {r:.4f}, r² = {r2:.4f}')

        # ----------------------------------------
        # RMSE (normalized ONLY here)
        # ----------------------------------------
        env_norm = (env - np.mean(env)) / np.std(env)
        pred_norm = (pred - np.mean(pred)) / np.std(pred)

        rmse = np.sqrt(np.mean((env_norm - pred_norm)**2))

        rmse_values[model].append(rmse)

        print(f'Normalized RMSE = {rmse:.4f}')
    

print('\n===== Universal decoder summary =====')

for model in models:
    mean_r = np.mean(r_values_universal[model])
    std_r = np.std(r_values_universal[model])
    mean_r2 = np.mean(r2_values_universal[model])
    std_r2 = np.std(r2_values_universal[model])
    mean_rmse = np.mean(rmse_values[model])
    std_rmse = np.std(rmse_values[model])

    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f}, '
          f'r² = {mean_r2:.4f} ± {std_r2:.4f}, '
          f'RMSE = {mean_rmse:.4f} ± {std_rmse:.4f}')



===== Universal decoder: envelope_log =====

Subject: S05
r = 0.0230, r² = 0.0005
Normalized RMSE = 1.3987

Subject: S34
r = 0.0975, r² = 0.0095
Normalized RMSE = 1.3908

Subject: S35
r = 0.0518, r² = 0.0027
Normalized RMSE = 1.4021

Subject: S03
r = 0.0563, r² = 0.0032
Normalized RMSE = 1.3863

Subject: S04
r = 0.0256, r² = 0.0007
Normalized RMSE = 1.3975

Subject: S44
r = 0.0267, r² = 0.0007
Normalized RMSE = 1.3991

Subject: S26
r = 0.0065, r² = 0.0000
Normalized RMSE = 1.4099

Subject: S19
r = 0.0932, r² = 0.0087
Normalized RMSE = 1.3721

Subject: S21
r = 0.1122, r² = 0.0126
Normalized RMSE = 1.3686

Subject: S17
r = 0.0640, r² = 0.0041
Normalized RMSE = 1.3892

Subject: S10
r = -0.0052, r² = 0.0000
Normalized RMSE = 1.4109

Subject: S42
r = 0.0805, r² = 0.0065
Normalized RMSE = 1.4062

Subject: S45
r = 0.0072, r² = 0.0001
Normalized RMSE = 1.4124

Subject: S11
r = 0.0348, r² = 0.0012
Normalized RMSE = 1.3957

Subject: S16
r = 0.1185, r² = 0.0140
Normalized RMSE = 1.3573

Subject:

In [12]:
# ------------------------------------------------
# Storage
# ------------------------------------------------
r_values_universal = {model: [] for model in models}
r2_values_universal = {model: [] for model in models}

# ------------------------------------------------
# Loop over models
# ------------------------------------------------
for model in models:

    print(f'\n===== Universal decoder: {model} =====')

    # ----------------------------------------
    # Load universal decoder TRF
    # ----------------------------------------
    trf_universal = eelbrain.load.unpickle(
        TRF_DIR / f'decoder-universal-trf-{model}.pickle'
    )

    # ----------------------------------------
    # Loop over subjects
    # ----------------------------------------
    for subject in SUBJECTS:

        print(f'\nSubject: {subject}')

        eeg = eelbrain.load.unpickle(
            STIMULUS_DIR / f'{subject}concatenated_eeg.pickle'
        )

        predictors = subject_model_predictors[subject][model]

        # ----------------------------------------
        # Predict stimulus from EEG
        # ----------------------------------------
        predicted = eelbrain.convolve(trf_universal, eeg)

        # ----------------------------------------
        # Build actual stimulus
        # ----------------------------------------
        #stimulus = sum(predictors) if len(predictors) > 1 else predictors[0]
        stimulus = predictors[0]

        # ----------------------------------------
        # Convert to numpy
        # ----------------------------------------
        env = stimulus.x
        pred = predicted.x

        # ----------------------------------------
        # Correlation
        # ----------------------------------------
        r = np.corrcoef(env, pred)[0,1]
        r2 = r**2

        r_values_universal[model].append(r)
        r2_values_universal[model].append(r2)

        print(f'r = {r:.4f}, r² = {r2:.4f}')

        # ----------------------------------------
        # RMSE
        # ----------------------------------------
        rmse_values = {model: [] for model in models}

        rmse = np.sqrt(np.mean((env - pred)**2))
        rmse_values[model].append(rmse)

        print(f'RMSE = {rmse:.4f}')

# ------------------------------------------------
# Summary statistics
# ------------------------------------------------
print('\n===== Universal decoder summary =====')

for model in models:
    mean_r = np.mean(r_values_universal[model])
    std_r = np.std(r_values_universal[model])
    mean_r2 = np.mean(r2_values_universal[model])
    std_r2 = np.std(r2_values_universal[model])
    mean_rmse = np.mean(rmse_values[model])
    std_rmse = np.std(rmse_values[model])

    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f}, r² = {mean_r2:.4f} ± {std_r2:.4f}, RMSE = {mean_rmse:.4f} ± {std_rmse:.4f}')



===== Universal decoder: envelope_log =====

Subject: S05
r = 0.0230, r² = 0.0005
RMSE = 124.6295

Subject: S34
r = 0.0975, r² = 0.0095
RMSE = 125.3754

Subject: S35
r = 0.0518, r² = 0.0027
RMSE = 125.3789

Subject: S03
r = 0.0563, r² = 0.0032
RMSE = 124.5998

Subject: S04
r = 0.0256, r² = 0.0007
RMSE = 124.6283

Subject: S44
r = 0.0267, r² = 0.0007
RMSE = 124.6083

Subject: S26
r = 0.0065, r² = 0.0000
RMSE = 128.4647

Subject: S19
r = 0.0932, r² = 0.0087
RMSE = 124.6106

Subject: S21
r = 0.1122, r² = 0.0126
RMSE = 124.6206

Subject: S17
r = 0.0640, r² = 0.0041
RMSE = 124.6164

Subject: S10
r = -0.0052, r² = 0.0000
RMSE = 124.6334

Subject: S42
r = 0.0805, r² = 0.0065
RMSE = 124.6280

Subject: S45
r = 0.0072, r² = 0.0001
RMSE = 124.7807

Subject: S11
r = 0.0348, r² = 0.0012
RMSE = 124.6246

Subject: S16
r = 0.1185, r² = 0.0140
RMSE = 124.6062

Subject: S20
r = 0.0411, r² = 0.0017
RMSE = 124.6179

Subject: S18
r = 0.0849, r² = 0.0072
RMSE = 124.6227

Subject: S01
r = 0.1571, r² = 0.024